# 01 — Data Exploration

**Purpose**: Understand the OASIS-2 longitudinal dataset before any modelling.

All logic lives in `src/data_exploration.py`. This notebook is a thin runner that calls those functions and displays results inline.

**What we cover here**:
- Dataset shape and column types
- Visit count distribution per participant
- Converter identification and conversion timing
- MMSE and nWBV trajectory plots
- Missing value audit
- Baseline distributions (how similar converters look at visit 1)

In [ ]:
import sys
from pathlib import Path

# Make sure src/ is importable when running from the notebooks/ directory
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib inline
import matplotlib.pyplot as plt

from src.data_exploration import (
    load_raw_dataset,
    summarise_dataset_structure,
    compute_visit_counts_per_participant,
    summarise_visit_distribution,
    identify_converters,
    summarise_converters,
    audit_missing_values,
    plot_visit_count_distribution,
    plot_mmse_trajectories_by_group,
    plot_converter_mmse_trajectories,
    plot_nwbv_trajectories_by_group,
    plot_baseline_distributions,
)

RAW_DATA_PATH = project_root / "data" / "raw" / "oasis_longitudinal.csv"
FIGURES_DIR   = project_root / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Data path    : {RAW_DATA_PATH}")
print(f"Figures dir  : {FIGURES_DIR}")

## 1. Load dataset

In [ ]:
oasis_df = load_raw_dataset(RAW_DATA_PATH)
oasis_df.head(10)

## 2. Dataset structure — dtypes, nulls, group distribution

In [ ]:
summarise_dataset_structure(oasis_df)

## 3. Visit count distribution

In [ ]:
visit_counts = compute_visit_counts_per_participant(oasis_df)
summarise_visit_distribution(visit_counts)

In [ ]:
plot_visit_count_distribution(visit_counts, FIGURES_DIR)
plt.show()

## 4. Converter identification

The 14 converters are the clinically critical group — participants who were nondemented at baseline but developed AD during the study. These are the participants our models need to flag early.

In [ ]:
converters_df = identify_converters(oasis_df)
summarise_converters(converters_df)

## 5. Missing value audit

In [ ]:
null_summary = audit_missing_values(oasis_df)
null_summary

## 6. MMSE trajectories by group

Each line is one participant. Converters (amber) are drawn on top of the background groups. The key observation: at early visits, converter MMSE scores often sit in the same range as non-demented participants — the signal is in the *rate and pattern of change*, not the absolute value.

In [ ]:
plot_mmse_trajectories_by_group(oasis_df, FIGURES_DIR)
plt.show()

## 7. Converter MMSE trajectories (individual)

One subplot per converter. The dashed red line marks the first visit where conversion was recorded in the dataset. Note how many converters show MMSE decline *before* the formal conversion label appears.

In [ ]:
plot_converter_mmse_trajectories(oasis_df, converters_df, FIGURES_DIR)
plt.show()

## 8. Brain volume (nWBV) trajectories by group

nWBV is the physical atrophy signal — it declines over time in all participants, but the slope is steeper in those with dementia. This is the signal that an LLM cannot reason about from a text description: it requires learning the *rate of change* across visits in embedding space.

In [ ]:
plot_nwbv_trajectories_by_group(oasis_df, FIGURES_DIR)
plt.show()

## 9. Baseline distributions — visit 1 only

This is the core difficulty of the prediction task: at visit 1, converters look nearly identical to non-demented participants on both MMSE and nWBV. The prediction challenge is real.

In [ ]:
plot_baseline_distributions(oasis_df, FIGURES_DIR)
plt.show()

## 10. Summary notes for preprocessing step

Record observations here before moving to `02_preprocessing.ipynb`.

In [ ]:
n_participants       = oasis_df["Subject ID"].nunique()
n_converters         = len(converters_df)
n_with_3plus_visits  = (visit_counts >= 3).sum()
missing_mmse_count   = oasis_df["MMSE"].isnull().sum()

print("── Exploration summary ────────────────────────────────")
print(f"  Total participants          : {n_participants}")
print(f"  Total imaging sessions      : {len(oasis_df)}")
print(f"  Converters                  : {n_converters}")
print(f"  Participants with ≥3 visits  : {n_with_3plus_visits}")
print(f"  Missing MMSE values         : {missing_mmse_count}")
print(f"  Group labels present        : {sorted(oasis_df['Group'].unique())}")
print()
print("── Notes for preprocessing ────────────────────────────")
print("  - MMSE missing values → median imputation per group in preprocessing step")
print("  - Use visits 1 and 2 as prediction cutoff regardless of total visit count")
print("  - Ground truth label: CDR worsened at ANY visit after visit 2")
print("  - Stratify train/test split to ensure converters appear in test set")